# EasyNCO TSP Benchmark（标准版：init + iteration）——逐数据 × 逐方法的显式命令

本 notebook 的目标：

- **完全通过 EasyNCO 平台的命令行**来做评测（`python EasyNCO/eval.py ...`），不在 notebook 里手写“调用模型/环境”的逻辑。
- 对以下数据集里所有 TSP 数据文件，逐个方法跑一遍：
  - `EasyNCO/data/datasets/cross_distribution/`（TSP 部分）
  - `EasyNCO/data/datasets/test_dataset_tsp_uniform/`
- 每个 block 都是“**一个数据文件 × 一个方法**”的一条独立命令，方便你：
  - 单独跑某一个组合
  - 对齐/复现某一次结果（看 log）
  - 逐方法对比

评测口径：

- 本 notebook **不再强制 init-only**。每个方法会按其 `EasyNCO/settings/<method>_settings.yaml` 的默认流程执行（通常是 `initialization + iteration`）。
- 你之前遇到的 `TSPEnv.lib_data` 相关报错，通常来自仓库里另一份 `EasyNCO/` 目录的实现差异；这里统一用 `python EasyNCO/eval.py ...`（并在仓库根目录执行）即可。

checkpoint 位置：

- 这些方法的 ckpt 在仓库根目录的 `EasyNCO/pretrained/` 下。
- 因为我们运行的是 `python EasyNCO/eval.py`，所以命令里会显式覆盖：
  - `settings.test_loader.model_dirpath=pretrained`

输出位置（你跑完后主要看这里）：

- 每个“方法 × 数据文件”会生成一份 log：`results/benchmarks/tsp_datasets/logs/<method>/<dataset_key>.log`


## 0. 运行前自检（非常重要）

Notebook 通常位于 `benchmarks/` 目录里；而 EasyNCO 的命令需要在仓库根目录执行。

为避免“在 notebook 里跑 bash 时 CWD 不对”，本 notebook 的每个 `%%bash` cell 都会：

1. `git rev-parse --show-toplevel` 自动定位仓库根目录
2. `cd` 到仓库根目录后再执行 `python EasyNCO/eval.py ...`

如果你是在没有 `.git` 的环境（比如只拷了部分目录）运行，`git rev-parse` 会失败；这种情况下请手动把 notebook 的工作目录切到仓库根目录再跑。


In [ ]:
from pathlib import Path

repo = Path.cwd().resolve()
print('当前工作目录:', repo)
print('是否存在 EasyNCO/eval.py:', (repo / 'EasyNCO' / 'eval.py').exists())
print('数据根目录(应存在):', repo / 'EasyNCO' / 'data' / 'datasets')
print('预训练 ckpt 目录(应存在):', repo / 'EasyNCO' / 'pretrained')


## 1. 入口说明（我们用 `EasyNCO/eval.py`）

本 notebook 的所有测试都通过命令行调用：

- `python EasyNCO/eval.py ...`

它的关键行为（你需要知道的）：

1. **数据根目录固定为** `EasyNCO/data/datasets/`
   - 所以 `test_data_path=...` 必须写成相对这个目录的路径（例如 `cross_distribution/tsp1000_explosion.pkl`）。
2. checkpoint 默认从 `EasyNCO/pretrained/` 读取
   - notebook 里仍然会显式写 `settings.test_loader.model_dirpath=pretrained`，避免你环境里默认值不同导致加载错模型。

如果你是在别的机器/目录运行，只要保证 `EasyNCO/` 目录结构保持一致，这些命令就能直接跑。


## 2. 方法清单（本 notebook 默认会跑哪些）

本 notebook 默认包含这些（你最关心的）方法：

- `omni`, `lehd`, `elg`, `icam`, `dact`, `htsp`

说明：

- `dact`：本仓库只提供了 `TSP50/TSP100` 的 ckpt；遇到其它规模的数据会在对应 block 里写明 SKIP。
- `htsp`：小规模（N<200）在 lower-level 容易出 shape mismatch；对应 block 默认 SKIP。

如果你希望把 `deepaco / difusco / t2t` 等也加入并实际跑起来，我可以按“有 ckpt 的规模”补齐它们的命令 block。


## 3. `python EasyNCO/eval.py ...` 参数详解（建议先读一遍）

### (A) 与“跑哪个方法”相关

- `settings=<xxx_settings>`：选择一份方法模板（在 `EasyNCO/settings/<xxx_settings>.yaml`）。
- `model=<method_name>`：方法名（主要用于日志展示、以及少数方法内部分支）。
- `settings.test_loader.model_dirpath=pretrained` + `settings.test_loader.model_filename=*.ckpt`：指定要加载的 checkpoint（相对 `EasyNCO/` 目录）。

### (B) 与“跑哪个数据”相关

- `test_data_path=<relative_path>`：必须是 **相对 `EasyNCO/data/datasets/`** 的路径。
- `problem=tsp`：指定问题类型。
- `scale=N`：告诉 env 当前问题规模（一般与文件名里的 `tspN`/`test_tspN` 对齐）。

### (C) 与“算多少个实例”相关

- `episodes=<K>`：评测多少个实例（通常等于数据文件里实例数）。
- `batch_size=<B>`：一次 forward 的 batch。规模越大，建议 B 越小，避免 OOM。

### (D) 其它常用项

- `decoder_strategy=greedy`：通常测试用 greedy。
- `cuda=[0]` + `CUDA_VISIBLE_DEVICES=0`：指定 GPU。
- `seed=1234`：固定随机种子。


---

## Dataset: `cross_distribution/tsp1000_explosion.pkl`

- 规模（scale / N）= **1000**
- episodes（测试实例数）= **2000**
- 文件格式 = `pkl(list)`
- 备注：cross_distribution：通常用于分布外泛化（rotation/explosion）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/cross_distribution__tsp1000_explosion.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：cross_distribution/tsp1000_explosion.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_explosion.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on cross_distribution/tsp1000_explosion.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_explosion.pkl \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_explosion.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/cross_distribution__tsp1000_explosion.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/cross_distribution__tsp1000_explosion.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/cross_distribution__tsp1000_explosion.pkl.log"


---

## Dataset: `cross_distribution/tsp1000_rotation.pkl`

- 规模（scale / N）= **1000**
- episodes（测试实例数）= **2000**
- 文件格式 = `pkl(list)`
- 备注：cross_distribution：通常用于分布外泛化（rotation/explosion）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/cross_distribution__tsp1000_rotation.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：cross_distribution/tsp1000_rotation.pkl
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_rotation.pkl.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on cross_distribution/tsp1000_rotation.pkl (scale=1000, episodes=2000, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=cross_distribution/tsp1000_rotation.pkl \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=2000 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/cross_distribution__tsp1000_rotation.pkl.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/cross_distribution__tsp1000_rotation.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/cross_distribution__tsp1000_rotation.pkl.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/cross_distribution__tsp1000_rotation.pkl.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt`

- 规模（scale / N）= **100**
- episodes（测试实例数）= **10000**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=50, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=50 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=50, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=50 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=50, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=50 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：dact
# 数据：test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/dact_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "[run][eval] dact on test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt (scale=100, episodes=10000, batch_size=128, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=dact \
  problem=tsp \
  scale=100 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp100_nums10000_uniform.pt \
  settings=dact_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=dact_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=10000 \
  batch_size=128 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "SKIP: HTSP 的 lower-level group_size=200；当 N<200 时经常 shape mismatch，这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp100_nums10000_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt`

- 规模（scale / N）= **200**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=16, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt (scale=200, episodes=128, batch_size=64, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=200 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp200_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=64 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp200_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt`

- 规模（scale / N）= **500**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=8, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=32, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=32 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=8, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=8, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt (scale=500, episodes=128, batch_size=32, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=500 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp500_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=32 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp500_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt`

- 规模（scale / N）= **1000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=4, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt (scale=1000, episodes=128, batch_size=16, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=1000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp1000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=16 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp1000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt`

- 规模（scale / N）= **2000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=2, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=8, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=2, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=2, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt (scale=2000, episodes=128, batch_size=8, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=2000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp2000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=8 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp2000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt`

- 规模（scale / N）= **3000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt (scale=3000, episodes=128, batch_size=4, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=3000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp3000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=4 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp3000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt`

- 规模（scale / N）= **4000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=2, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt (scale=4000, episodes=128, batch_size=2, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=4000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp4000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=2 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp4000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt`

- 规模（scale / N）= **5000**
- episodes（测试实例数）= **128**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt (scale=5000, episodes=128, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=5000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp5000_nums128_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=128 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp5000_nums128_uniform.pt.log"


---

## Dataset: `test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt`

- 规模（scale / N）= **10000**
- episodes（测试实例数）= **16**
- 文件格式 = `pt(torch)`
- 备注：uniform：同分布随机数据（用于 sanity check / scalability）。

建议：

- 如果你只想先跑通流程，可以先跑 1~2 个数据文件（例如 N=100 / N=1000），确认日志输出正常后再全量跑。
- 这类大规模数据（N>=3000）很容易显存吃紧：
  - 先从 `batch_size=1` 开始
  - 再逐步加大 batch_size


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：omni
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/omni_tsp_maml_fomaml.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/omni"
echo "[run][eval] omni on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=omni \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=omni_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=omni_tsp_maml_fomaml.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/omni/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：lehd
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/lehd_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/lehd"
echo "[run][eval] lehd on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=lehd \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=lehd_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=lehd_tsp100.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/lehd/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：elg
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/elg_tsp100.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/elg"
echo "[run][eval] elg on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=elg \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=elg_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=elg_tsp100.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/elg/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：icam
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/icam_tsp_best.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/icam"
echo "[run][eval] icam on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=8)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=icam \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=icam_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=icam_tsp_best.ckpt \
  ++settings.env.aug_factor=8 \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/icam/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/dact"
echo "SKIP: dact 仅提供 TSP50/TSP100 ckpt；（仅提供少数规模 ckpt），其他规模这里直接跳过。" | tee "results/benchmarks/tsp_datasets/logs/dact/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"

# 方法：htsp
# 数据：test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt
# 评测口径：标准（按 settings 默认 init+iteration）
# ckpt：EasyNCO/pretrained/htsp_tsp1000.ckpt
# 日志：results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log

mkdir -p "results/benchmarks/tsp_datasets/logs/htsp"
echo "[run][eval] htsp on test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt (scale=10000, episodes=16, batch_size=1, aug_factor=1)"

PYTHONWARNINGS=ignore CUDA_VISIBLE_DEVICES=0 python EasyNCO/eval.py \
  mode=test \
  model=htsp \
  problem=tsp \
  scale=10000 \
  decoder_strategy=greedy \
  test_data_path=test_dataset_tsp_uniform/test_tsp10000_nums16_uniform.pt \
  settings=htsp_settings \
  settings.test_loader.model_dirpath=pretrained \
  settings.test_loader.model_filename=htsp_tsp1000.ckpt \
  ++settings.env.aug_factor=1 \
  seed=1234 \
  cuda=[0] \
  episodes=16 \
  batch_size=1 \
  2>&1 | tee "results/benchmarks/tsp_datasets/logs/htsp/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"

exit_code=${PIPESTATUS[0]}
echo "[exit_code] ${exit_code}"
exit 0


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/deepaco"
echo "SKIP: deepaco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/deepaco/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/difusco"
echo "SKIP: difusco 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/difusco/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"


In [ ]:
%%bash
set -o pipefail
REPO_ROOT="$(git rev-parse --show-toplevel)"
cd "${REPO_ROOT}"
mkdir -p "results/benchmarks/tsp_datasets/logs/t2t"
echo "SKIP: t2t 在 EasyNCO 的实现里，初始化阶段不直接输出 tour/cost；真正求解/解码在 iteration 阶段完成。默认不在本 notebook 中跑（如需可手动启用）。" | tee "results/benchmarks/tsp_datasets/logs/t2t/test_dataset_tsp_uniform__test_tsp10000_nums16_uniform.pt.log"
